# 01 · Coleta de dados — API REST da SpaceX

> **Atenção:** execute todas as células (Kernel → Restart & Run All) e salve o notebook **com as saídas** antes de enviar ao GitHub. O curso exige células de código e de saída visíveis.

## Objetivo
Coletar lançamentos do Falcon 9 pela API REST da SpaceX e montar um DataFrame.

In [1]:
import requests, pandas as pd, numpy as np
pd.set_option('display.max_columns', None)

In [2]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)
data = pd.json_normalize(response.json())
data.head()

200


,flight_number,date_utc,rocket,payloads,launchpad,cores
0,1,2010-06-04T18:45:00.000Z,5e9d0d95eda69973a809d1ec,[5eb0e4b5b6c3bb0006eeb1e5],5e9e4501f509094ba4566f84,[{'core': '5e9e289df35918033d3b2623'}]
1,2,2010-06-04T18:45:00.000Z,5e9d0d95eda69973a809d1ec,[5eb0e4b5b6c3bb0006eeb1e5],5e9e4501f509094ba4566f84,[{'core': '5e9e289df35918033d3b2623'}]
2,3,2010-06-04T18:45:00.000Z,5e9d0d95eda69973a809d1ec,[5eb0e4b5b6c3bb0006eeb1e5],5e9e4501f509094ba4566f84,[{'core': '5e9e289df35918033d3b2623'}]
3,4,2010-06-04T18:45:00.000Z,5e9d0d95eda69973a809d1ec,[5eb0e4b5b6c3bb0006eeb1e5],5e9e4501f509094ba4566f84,[{'core': '5e9e289df35918033d3b2623'}]


Mantemos apenas as colunas úteis e lançamentos com um único núcleo e uma única carga.

In [3]:
data = data[['rocket','payloads','launchpad','cores','flight_number','date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= pd.to_datetime('2020-11-13').date()]
data.head()

,rocket,payloads,launchpad,cores,flight_number,date_utc,date
0,5e9d0d95eda69973a809d1ec,5eb0e4b5b6c3bb0006eeb1e5,5e9e4501f509094ba4566f84,{'core': '5e9e289df35918033d3b2623'},1,2010-06-04T18:45:00.000Z,2010-06-04
1,5e9d0d95eda69973a809d1ec,5eb0e4b5b6c3bb0006eeb1e5,5e9e4501f509094ba4566f84,{'core': '5e9e289df35918033d3b2623'},2,2010-06-04T18:45:00.000Z,2010-06-04
2,5e9d0d95eda69973a809d1ec,5eb0e4b5b6c3bb0006eeb1e5,5e9e4501f509094ba4566f84,{'core': '5e9e289df35918033d3b2623'},3,2010-06-04T18:45:00.000Z,2010-06-04
3,5e9d0d95eda69973a809d1ec,5eb0e4b5b6c3bb0006eeb1e5,5e9e4501f509094ba4566f84,{'core': '5e9e289df35918033d3b2623'},4,2010-06-04T18:45:00.000Z,2010-06-04


## Funções auxiliares
Cada função consulta um endpoint da API e preenche uma lista global.

In [4]:
BoosterVersion, PayloadMass, Orbit = [], [], []
LaunchSite, Longitude, Latitude = [], [], []

def getBoosterVersion(data):
    for x in data['rocket']:
        BoosterVersion.append(requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        r = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
        Longitude.append(r['longitude']); Latitude.append(r['latitude']); LaunchSite.append(r['name'])

def getPayloadData(data):
    for load in data['payloads']:
        r = requests.get("https://api.spacexdata.com/v4/payloads/" + load).json()
        PayloadMass.append(r['mass_kg']); Orbit.append(r['orbit'])

In [5]:
getBoosterVersion(data); getLaunchSite(data); getPayloadData(data)

## DataFrame final e filtro do Falcon 9

In [6]:
launch_dict = {'FlightNumber': list(data['flight_number']), 'Date': list(data['date']),
               'BoosterVersion': BoosterVersion, 'PayloadMass': PayloadMass, 'Orbit': Orbit,
               'LaunchSite': LaunchSite, 'Longitude': Longitude, 'Latitude': Latitude}
df = pd.DataFrame(launch_dict)
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].reset_index(drop=True)
data_falcon9['FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean())
data_falcon9.to_csv('../data/dataset_part_1.csv', index=False)
data_falcon9.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,-80.577366,28.561857
1,2,2010-06-04,Falcon 9,525.000000,LEO,CCAFS SLC 40,-80.577366,28.561857
2,3,2010-06-04,Falcon 9,677.000000,ISS,CCAFS SLC 40,-80.577366,28.561857
3,4,2010-06-04,Falcon 9,500.000000,PO,VAFB SLC 4E,-120.610829,34.632093


> Se a API retornar dados diferentes do laboratório, use o arquivo JSON/CSV estático fornecido pelo curso para reproduzir os resultados.